# ALPR Colombia - Entrenar YOLOv8 con Multi-Dataset
Descarga y combina multiples datasets de placas colombianas para entrenar un mejor detector.

In [ ]:
# 1. Verificar GPU
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    !nvidia-smi --query-gpu=memory.total,memory.free --format=csv,noheader

In [ ]:
# 2. Instalar dependencias
!pip install -q ultralytics roboflow

---
## Configuracion
Elige qué datasets usar y los parametros de entrenamiento.

In [ ]:
# ─── DATASETS ───
# Marcar con True los datasets a incluir
USE_NUEVO_1 = True   # alo-fwpow/placas-gshtx v4
USE_NUEVO_2 = True   # placasp/placas-bpbge    v2
USE_NUEVO_3 = True   # usco-thj9e/placas-colombia-ixdpr v5
USE_ORIGINAL = False # placas-colombia/proyecto-placas v2 (el que ya entrenaste)

# ─── ENTRENAMIENTO ───
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640

print('Configuracion lista.')

In [ ]:
# 3. Descargar datasets desde Roboflow
from roboflow import Roboflow
import shutil, os

rf = Roboflow(api_key='bUpcZuNnrkKdqieWD0qb')

datasets = []
if USE_NUEVO_1:
    print('Descargando placas-gshtx v4...')
    d1 = rf.workspace('alo-fwpow').project('placas-gshtx').version(4).download('yolov8')
    datasets.append(('gshtx', d1.location))
    print(f'  -> {d1.location}')
if USE_NUEVO_2:
    print('Descargando placas-bpbge v2...')
    d2 = rf.workspace('placasp').project('placas-bpbge').version(2).download('yolov8')
    datasets.append(('bpbge', d2.location))
    print(f'  -> {d2.location}')
if USE_NUEVO_3:
    print('Descargando placas-colombia-ixdpr v5...')
    d3 = rf.workspace('usco-thj9e').project('placas-colombia-ixdpr').version(5).download('yolov8')
    datasets.append(('ixdpr', d3.location))
    print(f'  -> {d3.location}')
if USE_ORIGINAL:
    print('Descargando proyecto-placas v2...')
    d0 = rf.workspace('placas-colombia').project('proyecto-placas-8arfj').version(2).download('yolov8')
    datasets.append(('original', d0.location))
    print(f'  -> {d0.location}')

print(f'\nTotal datasets descargados: {len(datasets)}')

In [ ]:
# 4. Fusionar datasets en una sola estructura
import glob

COMBINED = '/content/placas-combinado'
os.makedirs(COMBINED, exist_ok=True)

total_images = 0
for prefix, loc in datasets:
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(loc, split, 'images')
        lbl_dir = os.path.join(loc, split, 'labels')
        if not os.path.isdir(img_dir):
            continue

        dst_img = os.path.join(COMBINED, split, 'images')
        dst_lbl = os.path.join(COMBINED, split, 'labels')
        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_lbl, exist_ok=True)

        imgs = glob.glob(os.path.join(img_dir, '*'))
        for src in imgs:
            fname = os.path.basename(src)
            new_name = f'{prefix}_{fname}'
            shutil.copy2(src, os.path.join(dst_img, new_name))

            # Copiar label correspondiente
            base, ext = os.path.splitext(fname)
            lbl_src = os.path.join(lbl_dir, base + '.txt')
            if os.path.exists(lbl_src):
                shutil.copy2(lbl_src, os.path.join(dst_lbl, f'{prefix}_{base}.txt'))

        count = len(imgs)
        total_images += count
        print(f'{prefix}/{split}: {count} imagenes')

print(f'\nTotal imagenes combinadas: {total_images}')

# Contar por split
for split in ['train', 'valid', 'test']:
    p = os.path.join(COMBINED, split, 'images')
    if os.path.isdir(p):
        print(f'  {split}: {len(os.listdir(p))} imagenes')

In [ ]:
# 5. Crear data.yaml combinado
import yaml

data_yaml = {
    'train': os.path.join(COMBINED, 'train', 'images'),
    'val':   os.path.join(COMBINED, 'valid', 'images'),
    'test':  os.path.join(COMBINED, 'test', 'images') if os.path.isdir(os.path.join(COMBINED, 'test', 'images')) else '',
    'nc': 1,
    'names': ['placa'],
}

yaml_path = os.path.join(COMBINED, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'data.yaml creado en: {yaml_path}')
!cat {yaml_path}

In [ ]:
# 6. Entrenar YOLOv8n con hiperparametros optimizados
from ultralytics import YOLO
import torch

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name='plate_detector',
    patience=30,
    save=True,
    plots=True,
    # Aumentacion de datos (optimizada para placas)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.1,
    mosaic=1.0,
    mixup=0.1,
    # Optimizador
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    # Gestion de recursos
    workers=4,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

In [ ]:
# 7. Evaluar modelo final
metrics = model.val(data=yaml_path)
print(f'\nResultados:')
print(f'  mAP50:    {metrics.box.map50:.3f}')
print(f'  mAP50-95: {metrics.box.map:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}')
print(f'  Recall:   {metrics.box.mr:.3f}')

In [ ]:
# 8. Exportar modelo para descargar
import shutil, os

best_path = '/content/runs/detect/plate_detector/weights/best.pt'
last_path = '/content/runs/detect/plate_detector/weights/last.pt'

if os.path.exists(best_path):
    shutil.copy(best_path, '/content/plate_detector.pt')
    print(f'Modelo exportado: /content/plate_detector.pt')

shutil.make_archive('/content/plate_detector_model', 'zip', '/content/runs/detect/plate_detector')
print(f'Zip completo: /content/plate_detector_model.zip')

import os
pt_size = os.path.getsize('/content/plate_detector.pt') / 1e6 if os.path.exists('/content/plate_detector.pt') else 0
zip_size = os.path.getsize('/content/plate_detector_model.zip') / 1e6
print(f'\nTamanos:')
print(f'  plate_detector.pt:       {pt_size:.1f} MB')
print(f'  plate_detector_model.zip: {zip_size:.1f} MB')

In [ ]:
# 9. Descargar a tu PC
from google.colab import files
files.download('/content/plate_detector.pt')